In [ ]:
data_file: str = "../samples/bny/DependencyVulnerabilityCheck_VulnerabilityReport.csv"
enrichment_file: str = "../samples/bny/enrichment_data.json"
repository_filter: list[str] = []
exclude_log4j: bool = True

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.simplefilter("ignore")

# Load vulnerability data
df = pd.read_csv(data_file)

# Load enrichment data (EPSS and KEV)
try:
    with open(enrichment_file) as f:
        enrichment = json.load(f)
    epss_data = enrichment.get('epss', {})
    kev_data = enrichment.get('kev', {})
except:
    epss_data = {}
    kev_data = {}

# Filter repositories if specified
if len(repository_filter) > 0:
    df = df[df['repositoryPath'].str.contains('|'.join(repository_filter), case=False)]

# Exclude Log4j if specified
log4j_cves = ['CVE-2021-44228', 'CVE-2021-45046', 'CVE-2021-4104', 'CVE-2019-17571',
              'CVE-2022-23305', 'CVE-2022-23307', 'CVE-2022-23302', 'CVE-2023-26464']
if exclude_log4j:
    df = df[~df['cve'].isin(log4j_cves)]

In [ ]:
# Build enriched CVE analysis
severity_map = {'CRITICAL': 4, 'HIGH': 3, 'MODERATE': 2, 'LOW': 1}

cve_analysis = []
for cve in df['cve'].unique():
    cve_df = df[df['cve'] == cve]
    epss = epss_data.get(cve, {})
    kev = kev_data.get(cve, {})
    
    cve_analysis.append({
        'cve': cve,
        'severity': cve_df['severity'].iloc[0],
        'severity_score': severity_map.get(cve_df['severity'].iloc[0], 0),
        'groupId': cve_df['groupId'].iloc[0],
        'artifactId': cve_df['artifactId'].iloc[0],
        'artifact': f"{cve_df['groupId'].iloc[0]}:{cve_df['artifactId'].iloc[0]}",
        'summary': cve_df['summary'].iloc[0],
        'repos_affected': cve_df['repositoryPath'].nunique(),
        'projects_affected': cve_df['projectName'].nunique(),
        'cwes': str(cve_df['CWEs'].iloc[0]),
        'epss_score': epss.get('epss_score', 0),
        'epss_percentile': epss.get('epss_percentile', 0),
        'in_kev': kev.get('in_kev', False),
        'ransomware_use': kev.get('ransomware_use', 'N/A'),
        'exploit_count': kev.get('exploit_count', 0),
        'exploitation_reports': kev.get('exploitation_reports', 0)
    })

analysis_df = pd.DataFrame(cve_analysis)

# Calculate composite risk score
# Weight: KEV presence (30%), EPSS (25%), Severity (20%), Spread (15%), Exploits (10%)
analysis_df['risk_score'] = (
    analysis_df['in_kev'].astype(int) * 30 +
    analysis_df['epss_score'] * 25 +
    (analysis_df['severity_score'] / 4) * 20 +
    (analysis_df['repos_affected'] / analysis_df['repos_affected'].max()) * 15 +
    (analysis_df['exploit_count'].clip(upper=50) / 50) * 10
)

analysis_df = analysis_df.sort_values('risk_score', ascending=False)

In [ ]:
# Create Risk Matrix Visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Exploitability vs Severity (Size = Repos Affected)',
        'Top 15 Highest Risk CVEs',
        'Vulnerabilities by Artifact',
        'Risk Distribution by Category'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'bar'}, {'type': 'pie'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Color mapping for severity
severity_colors = {'CRITICAL': '#FF5B5B', 'HIGH': '#FABA49', 'MODERATE': '#FEE968', 'LOW': '#52BBA0'}

# 1. Scatter plot: EPSS vs Severity with KEV highlighting
for severity in ['CRITICAL', 'HIGH', 'MODERATE', 'LOW']:
    sev_df = analysis_df[analysis_df['severity'] == severity]
    
    # Non-KEV vulnerabilities
    non_kev = sev_df[sev_df['in_kev'] == False]
    if len(non_kev) > 0:
        fig.add_trace(
            go.Scatter(
                x=non_kev['epss_score'],
                y=non_kev['severity_score'] + (np.random.rand(len(non_kev)) - 0.5) * 0.3,
                mode='markers',
                marker=dict(
                    size=non_kev['repos_affected'] * 2 + 5,
                    color=severity_colors[severity],
                    opacity=0.6,
                    line=dict(width=1, color='white')
                ),
                name=severity,
                text=non_kev['cve'],
                customdata=non_kev[['artifact', 'summary', 'repos_affected', 'epss_percentile']].values,
                hovertemplate='<b>%{text}</b><br>Artifact: %{customdata[0]}<br>EPSS: %{x:.3f} (%{customdata[3]:.1%}ile)<br>Repos: %{customdata[2]}<br>%{customdata[1]}<extra></extra>',
                legendgroup=severity,
                showlegend=True
            ),
            row=1, col=1
        )
    
    # KEV vulnerabilities (with star marker)
    kev = sev_df[sev_df['in_kev'] == True]
    if len(kev) > 0:
        fig.add_trace(
            go.Scatter(
                x=kev['epss_score'],
                y=kev['severity_score'] + (np.random.rand(len(kev)) - 0.5) * 0.3,
                mode='markers',
                marker=dict(
                    size=kev['repos_affected'] * 2 + 8,
                    color=severity_colors[severity],
                    symbol='star',
                    line=dict(width=2, color='black')
                ),
                name=f'{severity} (KEV)',
                text=kev['cve'],
                customdata=kev[['artifact', 'summary', 'repos_affected', 'epss_percentile', 'exploit_count']].values,
                hovertemplate='<b>%{text}</b> (IN KEV)<br>Artifact: %{customdata[0]}<br>EPSS: %{x:.3f} (%{customdata[3]:.1%}ile)<br>Repos: %{customdata[2]}<br>Exploits: %{customdata[4]}<br>%{customdata[1]}<extra></extra>',
                legendgroup=severity,
                showlegend=True
            ),
            row=1, col=1
        )

# 2. Top 15 Risk Bar Chart
top15 = analysis_df.head(15)
fig.add_trace(
    go.Bar(
        y=top15['cve'],
        x=top15['risk_score'],
        orientation='h',
        marker=dict(
            color=[severity_colors.get(s, '#999') for s in top15['severity']],
            line=dict(width=2, color=['black' if k else 'white' for k in top15['in_kev']])
        ),
        text=[f"{'KEV ' if k else ''}{s[:4]}" for k, s in zip(top15['in_kev'], top15['severity'])],
        textposition='inside',
        customdata=top15[['artifact', 'repos_affected', 'epss_score']].values,
        hovertemplate='<b>%{y}</b><br>Risk Score: %{x:.1f}<br>Artifact: %{customdata[0]}<br>Repos: %{customdata[1]}<br>EPSS: %{customdata[2]:.3f}<extra></extra>',
        showlegend=False
    ),
    row=1, col=2
)

# 3. Vulnerabilities by Artifact
artifact_vulns = analysis_df.groupby('artifactId').agg({
    'cve': 'nunique',
    'repos_affected': 'max',
    'in_kev': 'sum'
}).sort_values('cve', ascending=True).tail(12)

fig.add_trace(
    go.Bar(
        y=artifact_vulns.index,
        x=artifact_vulns['cve'],
        orientation='h',
        marker=dict(
            color='#4A90D9',
            line=dict(width=1, color='white')
        ),
        text=[f"{v} CVEs ({k} KEV)" if k > 0 else f"{v} CVEs" for v, k in zip(artifact_vulns['cve'], artifact_vulns['in_kev'])],
        textposition='inside',
        showlegend=False
    ),
    row=2, col=1
)

# 4. Risk Category Pie Chart
risk_categories = {
    'KEV + Critical EPSS': len(analysis_df[(analysis_df['in_kev'] == True) & (analysis_df['epss_percentile'] > 0.9)]),
    'KEV Only': len(analysis_df[(analysis_df['in_kev'] == True) & (analysis_df['epss_percentile'] <= 0.9)]),
    'High EPSS (>90%ile)': len(analysis_df[(analysis_df['in_kev'] == False) & (analysis_df['epss_percentile'] > 0.9)]),
    'Moderate EPSS (50-90%ile)': len(analysis_df[(analysis_df['in_kev'] == False) & (analysis_df['epss_percentile'] > 0.5) & (analysis_df['epss_percentile'] <= 0.9)]),
    'Low EPSS (<50%ile)': len(analysis_df[(analysis_df['in_kev'] == False) & (analysis_df['epss_percentile'] <= 0.5)])
}

fig.add_trace(
    go.Pie(
        labels=list(risk_categories.keys()),
        values=list(risk_categories.values()),
        marker=dict(colors=['#FF5B5B', '#FABA49', '#FEE968', '#A8D5BA', '#52BBA0']),
        textinfo='label+value',
        hole=0.3
    ),
    row=2, col=2
)

# Update layout
fig.update_xaxes(title_text='EPSS Score (Exploit Probability)', row=1, col=1)
fig.update_yaxes(
    title_text='Severity',
    ticktext=['LOW', 'MODERATE', 'HIGH', 'CRITICAL'],
    tickvals=[1, 2, 3, 4],
    row=1, col=1
)
fig.update_xaxes(title_text='Risk Score', row=1, col=2)
fig.update_xaxes(title_text='Unique CVEs', row=2, col=1)

fig.update_layout(
    height=900,
    title=dict(
        text='<b>Vulnerability Risk Prioritization Dashboard</b><br><sup>Stars indicate CVEs in CISA/VulnCheck KEV (Known Exploited)</sup>',
        x=0.5
    ),
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.show(renderer='plotly_mimetype')

In [ ]:
# Create detailed priority table
priority_df = analysis_df.head(25)[[
    'cve', 'severity', 'artifactId', 'repos_affected',
    'epss_score', 'epss_percentile', 'in_kev', 'ransomware_use',
    'exploit_count', 'risk_score', 'summary'
]].copy()

priority_df['epss_display'] = priority_df.apply(
    lambda r: f"{r['epss_score']:.3f} ({r['epss_percentile']*100:.0f}%ile)", axis=1
)
priority_df['kev_status'] = priority_df.apply(
    lambda r: f"KEV ({r['exploit_count']} POCs)" if r['in_kev'] else 'No', axis=1
)
priority_df['risk_score'] = priority_df['risk_score'].round(1)

# Create styled table
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>CVE</b>', '<b>Severity</b>', '<b>Artifact</b>', '<b>Repos</b>',
                '<b>EPSS</b>', '<b>KEV Status</b>', '<b>Risk</b>', '<b>Summary</b>'],
        fill_color='#2E4057',
        font=dict(color='white', size=11),
        align='left',
        height=30
    ),
    cells=dict(
        values=[
            priority_df['cve'],
            priority_df['severity'],
            priority_df['artifactId'],
            priority_df['repos_affected'],
            priority_df['epss_display'],
            priority_df['kev_status'],
            priority_df['risk_score'],
            [s[:60] + '...' if len(str(s)) > 60 else s for s in priority_df['summary']]
        ],
        fill_color=[
            ['#FFECEC' if s == 'CRITICAL' else '#FFF4E0' if s == 'HIGH' else '#FFFCE0' if s == 'MODERATE' else '#E8F5F0'
             for s in priority_df['severity']]
        ] * 8,
        font=dict(size=10),
        align='left',
        height=25
    )
)])

fig_table.update_layout(
    title='<b>Top 25 Priority Vulnerabilities</b><br><sup>Ranked by composite risk score (KEV status, EPSS, severity, spread)</sup>',
    height=700,
    margin=dict(l=10, r=10, t=60, b=10)
)

fig_table.show(renderer='plotly_mimetype')

In [ ]:
# Create CWE Analysis Treemap
# Explode CWEs and count
cwe_list = []
for _, row in analysis_df.iterrows():
    cwes = str(row['cwes']).split(';')
    for cwe in cwes:
        cwe = cwe.strip()
        if cwe and cwe != 'nan':
            cwe_list.append({
                'cwe': cwe,
                'severity': row['severity'],
                'in_kev': row['in_kev']
            })

cwe_df = pd.DataFrame(cwe_list)
cwe_counts = cwe_df.groupby('cwe').agg({
    'severity': 'count',
    'in_kev': 'sum'
}).reset_index()
cwe_counts.columns = ['cwe', 'count', 'kev_count']
cwe_counts = cwe_counts.sort_values('count', ascending=False).head(20)

# Map CWE IDs to descriptions
cwe_descriptions = {
    'CWE-502': 'Deserialization of Untrusted Data',
    'CWE-400': 'Uncontrolled Resource Consumption',
    'CWE-770': 'Allocation Without Limits',
    'CWE-20': 'Improper Input Validation',
    'CWE-787': 'Out-of-bounds Write',
    'CWE-22': 'Path Traversal',
    'CWE-674': 'Uncontrolled Recursion',
    'CWE-611': 'XXE (XML External Entity)',
    'CWE-200': 'Information Exposure',
    'CWE-918': 'Server-Side Request Forgery',
    'CWE-601': 'Open Redirect',
    'CWE-917': 'Expression Language Injection',
    'CWE-94': 'Code Injection',
    'CWE-285': 'Improper Authorization',
    'CWE-74': 'Injection',
    'CWE-121': 'Stack-based Buffer Overflow',
    'CWE-835': 'Infinite Loop',
    'CWE-178': 'Case Sensitivity Issues',
    'CWE-434': 'Unrestricted Upload',
    'CWE-89': 'SQL Injection'
}

cwe_counts['description'] = cwe_counts['cwe'].map(lambda x: cwe_descriptions.get(x, x))
cwe_counts['label'] = cwe_counts.apply(lambda r: f"{r['cwe']}<br>{r['description']}", axis=1)

fig_cwe = px.treemap(
    cwe_counts,
    path=['label'],
    values='count',
    color='kev_count',
    color_continuous_scale='Reds',
    title='<b>Top 20 Weakness Types (CWEs)</b><br><sup>Color intensity = number of CVEs in KEV</sup>'
)

fig_cwe.update_layout(height=500)
fig_cwe.show(renderer='plotly_mimetype')

In [ ]:
# Summary Statistics
print("=" * 80)
print("VULNERABILITY RISK SUMMARY")
print("=" * 80)
print(f"\nTotal unique CVEs analyzed: {len(analysis_df)}")
print(f"CVEs in KEV (confirmed exploited): {analysis_df['in_kev'].sum()}")
print(f"CVEs with ransomware association: {len(analysis_df[analysis_df['ransomware_use'] == 'Known'])}")
print(f"\nSeverity Distribution:")
for sev in ['CRITICAL', 'HIGH', 'MODERATE', 'LOW']:
    count = len(analysis_df[analysis_df['severity'] == sev])
    kev_count = len(analysis_df[(analysis_df['severity'] == sev) & (analysis_df['in_kev'] == True)])
    print(f"  {sev}: {count} ({kev_count} in KEV)")

print(f"\nHigh-Risk CVEs (EPSS > 90th percentile): {len(analysis_df[analysis_df['epss_percentile'] > 0.9])}")
print(f"CVEs with known exploit POCs: {len(analysis_df[analysis_df['exploit_count'] > 0])}")